In [152]:
import upload.pdf_loader as p
print(dir(p))

['Loader_', 'PyPDFLoader', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__']


In [158]:
import splitter.splitters as p
print(dir(p))

['RecursiveCharacterTextSplitter', 'Split_Text', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__']


In [140]:
import sys
import importlib

sys.path.append(r"D:\AI Research Paper Simplifier")

import PROMPT.rag_prompt as rp

importlib.reload(rp)

print(rp.RAG_PROMPT)

d:\AI Research Paper Simplifier
input_variables=['context', 'question'] input_types={} partial_variables={} template='\nYou are an advanced AI Research Paper Assistant.\n\nYou help students understand research papers, IEEE journals, and academic documents.\n\nYou are given CONTEXT retrieved from uploaded documents and academic sources.\n\n---\n\nSTRICT RULES:\n1. Use ONLY the provided context.\n2. Do NOT use outside knowledge.\n3. If information is missing, say: "Not clearly mentioned in the paper."\n4. Do NOT hallucinate.\n\n---\n\nTASKS:\n\n1. Simplify research paper for BSCS students.\n2. Explain equations step-by-step.\n3. Extract methodology (model, dataset, pipeline, evaluation).\n4. Describe architecture visually in text form.\n5. Convert into PPT-style content.\n6. Provide citations if available.\n\n---\n\nCONTEXT:\n{context}\n\n---\n\nQUESTION:\n{question}\n\n---\n\nOUTPUT FORMAT:\n\nSimple Explanation:\n...\n\nTechnical Details:\n...\n\nMethodology:\n...\n\nEquations:\n...\n\

In [159]:
from langchain_community.vectorstores import FAISS
from model.models import Model_
from PROMPT.rag_prompt import RAG_PROMPT
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence,RunnablePassthrough,RunnableParallel,RunnableLambda
from upload.pdf_loader import Loader_
from splitter.splitters import Split_Text
from dotenv import load_dotenv
load_dotenv()

True

In [160]:
paper=r"D:\AI Research Paper Simplifier\Lecun2015.pdf"
docs=Loader_.load_pdf(paper)

In [161]:
print(docs[2].page_content)

be seen as a kind of hilly landscape in the high-dimensional space of 
weight values. The negative gradient vector indicates the direction 
of steepest descent in this landscape, taking it closer to a minimum, 
where the output error is low on average. 
In practice, most practitioners use a procedure called stocha stic 
gradient descent (SGD). This consists of showing the input vector 
for a few examples, computing the outputs and the errors, computing 
the average gradient for those examples, and adjusting the weights 
accordingly. The process is repeated for many small sets of examples 
from the training set until the average of the objective function stops 
decreasing. It is called stochastic because each small set of examples 
gives a noisy estimate of the average gradient over all examples. This 
simple procedure usually finds a good set of weights surprisingly 
quickly when compared with far more elaborate optimization tech-
niques18. After training, the performance of the system

In [165]:
chunk=Split_Text(docs)
len(chunk)

84

In [166]:
vector=FAISS.from_documents(chunk,Model_.embeddings())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 832.68it/s]


In [167]:
len(vector.index_to_docstore_id) 

84

In [168]:
retriever1=vector.as_retriever(
    search_type='mmr',
    search_kwargs={"k":6, "lambda_mult":0}
)

In [ ]:
# for i , doc in enumerate(result):
#     print(f"\n---result {i+1}---\n")
#     print(f"content :\n {doc.page_content}")


---result 1---

content :
 a sorted list of symbols when their input consists of an unsorted 
sequence in which each symbol is accompanied by a real value that 
indicates its priority in the list 88. Memory networks can be trained 
to keep track of the state of the world in a setting similar to a text 
adventure game and after reading a story, they can answer questions 
that require complex inference90. In one test example, the network is 
shown a 15-sentence version of the The Lord of the Rings and correctly 
answers questions such as “where is Frodo now?”89.  
The future of deep learning 
Unsupervised learning91–98 had a catalytic effect in reviving interest in 
deep learning, but has since been overshadowed by the successes of 
purely supervised learning. Although we have not focused on it in this 
Review, we expect unsupervised learning to become far more important 
in the longer term. Human and animal learning is largely unsupervised: 
we discover the structure of the world by ob

In [169]:
def format_doc(retrieve_docs):
    context_str = "\n\n".join([d.page_content for d in retrieve_docs])
    return context_str

In [170]:
runnable=RunnableParallel({
    'context':retriever1 |RunnableLambda(format_doc),
    'question':RunnablePassthrough()
    
})

In [171]:
parser=StrOutputParser()

In [172]:
response_chain= RAG_PROMPT | Model_.groqchat_model() | parser

In [173]:
chain=runnable | response_chain
chain.invoke("Explain the methodology of this paper.")

'Simple Explanation:\nThe paper discusses a methodology for machine translation using deep learning techniques. It explains how a machine can be trained to generate sequences of French words based on a given English sentence. The process involves using a decoder network to output a probability distribution for each word in the translation.\n\nTechnical Details:\nThe paper mentions that the machine is trained using a large dataset of labelled examples, with hundreds of millions of adjustable weights. The learning algorithm computes a gradient vector to adjust the weight vector and minimize the error between the output scores and the desired pattern of scores.\n\nMethodology:\nThe methodology used in this paper involves the following steps:\n1. **Data Preparation**: The paper uses a large dataset of labelled examples for training the machine.\n2. **Model**: The model used is a deep learning system, specifically a decoder network, to generate sequences of French words.\n3. **Pipeline**: T